In [1]:
import requests

# Where USD is the base currency you want to use
url = 'https://v6.exchangerate-api.com/v6/834ef49e34cbb28319e9c4be/latest/USD'

# Making our request
response = requests.get(url)
data = response.json()

# Your JSON object
print(data)
		

{'result': 'success', 'documentation': 'https://www.exchangerate-api.com/docs', 'terms_of_use': 'https://www.exchangerate-api.com/terms', 'time_last_update_unix': 1773705601, 'time_last_update_utc': 'Tue, 17 Mar 2026 00:00:01 +0000', 'time_next_update_unix': 1773792001, 'time_next_update_utc': 'Wed, 18 Mar 2026 00:00:01 +0000', 'base_code': 'USD', 'conversion_rates': {'USD': 1, 'AED': 3.6725, 'AFN': 62.9554, 'ALL': 83.8637, 'AMD': 377.6034, 'ANG': 1.79, 'AOA': 922.9185, 'ARS': 1452.25, 'AUD': 1.4169, 'AWG': 1.79, 'AZN': 1.6982, 'BAM': 1.7024, 'BBD': 2.0, 'BDT': 122.6398, 'BGN': 1.6421, 'BHD': 0.376, 'BIF': 2980.2845, 'BMD': 1.0, 'BND': 1.2787, 'BOB': 6.9129, 'BRL': 5.262, 'BSD': 1.0, 'BTN': 92.3996, 'BWP': 13.8033, 'BYN': 2.9633, 'BZD': 2.0, 'CAD': 1.3683, 'CDF': 2233.6851, 'CHF': 0.7882, 'CLF': 0.02322, 'CLP': 917.6553, 'CNH': 6.8933, 'CNY': 6.8983, 'COP': 3691.5807, 'CRC': 469.8936, 'CUP': 24.0, 'CVE': 95.9799, 'CZK': 21.2728, 'DJF': 177.721, 'DKK': 6.4981, 'DOP': 61.6195, 'DZD': 132

In [2]:
data

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1773705601,
 'time_last_update_utc': 'Tue, 17 Mar 2026 00:00:01 +0000',
 'time_next_update_unix': 1773792001,
 'time_next_update_utc': 'Wed, 18 Mar 2026 00:00:01 +0000',
 'base_code': 'USD',
 'conversion_rates': {'USD': 1,
  'AED': 3.6725,
  'AFN': 62.9554,
  'ALL': 83.8637,
  'AMD': 377.6034,
  'ANG': 1.79,
  'AOA': 922.9185,
  'ARS': 1452.25,
  'AUD': 1.4169,
  'AWG': 1.79,
  'AZN': 1.6982,
  'BAM': 1.7024,
  'BBD': 2.0,
  'BDT': 122.6398,
  'BGN': 1.6421,
  'BHD': 0.376,
  'BIF': 2980.2845,
  'BMD': 1.0,
  'BND': 1.2787,
  'BOB': 6.9129,
  'BRL': 5.262,
  'BSD': 1.0,
  'BTN': 92.3996,
  'BWP': 13.8033,
  'BYN': 2.9633,
  'BZD': 2.0,
  'CAD': 1.3683,
  'CDF': 2233.6851,
  'CHF': 0.7882,
  'CLF': 0.02322,
  'CLP': 917.6553,
  'CNH': 6.8933,
  'CNY': 6.8983,
  'COP': 3691.5807,
  'CRC': 469.8936,
  'CUP': 24.0,
  'CVE': 95

In [3]:
data.get('conversion_rates').get('EUR')

0.8706

In [4]:
%pip install openmeteo-requests
%pip install requests-cache retry-requests numpy pandas


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 41.3888,
	"longitude": 2.159,
	"current": "temperature_2m",
}
responses = openmeteo.weather_api(url, params=params)

print(responses[0].Current().Variables(0).Value())
# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process current data. The order of variables needs to be the same as requested.
current = response.Current()
current_temperature_2m = current.Variables(0).Value()

print(f"\nCurrent time: {current.Time()}")
print(f"Current temperature_2m: {current_temperature_2m}")


10.968499183654785
Coordinates: 41.375°N 2.125°E
Elevation: 44.0 m asl
Timezone difference to GMT+0: 0s

Current time: 1773787500
Current temperature_2m: 10.968499183654785


In [6]:
def convert_currency(amount, from_currency, to_currency):
    url = f'https://v6.exchangerate-api.com/v6/834ef49e34cbb28319e9c4be/pair/{from_currency}/{to_currency}'
    response = requests.get(url)
    data = response.json()
    
    if data['result'] == 'success':
        exchange_rate = data['conversion_rate']
        converted_amount = amount * exchange_rate
        print(f"{amount} {from_currency} is equal to {converted_amount:.2f} {to_currency} at an exchange rate of {exchange_rate:.4f}.")
        return converted_amount
    else:
        raise ValueError(f"Error fetching exchange rate: {data['error-type']}")

In [7]:
convert_currency(100, 'USD', 'EUR')

100 USD is equal to 87.06 EUR at an exchange rate of 0.8706.


87.06

In [8]:
def get_exchange_rate(from_currency, to_currency):
    url = f'https://v6.exchangerate-api.com/v6/834ef49e34cbb28319e9c4be/pair/{from_currency}/{to_currency}'
    response = requests.get(url)
    data = response.json()
    
    if data['result'] == 'success':
        exchange_rate = data['conversion_rate']
        print(f"Exchange rate from {from_currency} to {to_currency}: {exchange_rate:.4f}")
        return exchange_rate
    else:
        raise ValueError(f"Error fetching exchange rate: {data['error-type']}")

In [9]:
get_exchange_rate('USD', 'EUR')

Exchange rate from USD to EUR: 0.8706


0.8706

In [10]:
def geocode_city(city_name):
    url = "https://geocoding-api.open-meteo.com/v1/search"
    params = {
        "name": city_name,
        "count": 1,
        "language": "en",
        "format": "json",
    }
    response = requests.get(url, params=params, timeout=15)
    response.raise_for_status()
    data = response.json()

    results = data.get("results", [])
    if results:
        latitude = results[0]["latitude"]
        longitude = results[0]["longitude"]
        print(f"Coordinates of {city_name}: {latitude}°N, {longitude}°E")
        return latitude, longitude
    else:
        raise ValueError(f"City '{city_name}' not found.")

In [11]:
lat,long = geocode_city('Barcelona')

Coordinates of Barcelona: 41.38879°N, 2.15899°E


In [12]:
def get_current_temperature(latitude, longitude):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": "temperature_2m",
    }
    responses = openmeteo.weather_api(url, params=params)
    response = responses[0]
    current = response.Current()
    return current.Variables(0).Value()

In [13]:
get_current_temperature(lat, long)

10.968499183654785

In [14]:
def get_weather_forecast(latitude, longitude, days=7):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "daily": "temperature_2m_max,temperature_2m_min",
        "timezone": "auto",
        "forecast_days": days
    }
    responses = openmeteo.weather_api(url, params=params)
    response = responses[0]
    daily = response.Daily()

    temp_max = daily.Variables(0).ValuesAsNumpy()
    temp_min = daily.Variables(1).ValuesAsNumpy()
    n_days = len(temp_max)

    start_ts = daily.Time()
    step_seconds = daily.Interval()
    dates = pd.to_datetime(
        [start_ts + i * step_seconds for i in range(n_days)],
        unit="s",
        utc=True
    ).tz_convert(None)

    forecast_data = []
    for i in range(n_days):
        forecast_data.append({
            "date": dates[i],
            "temp_max": float(temp_max[i]),
            "temp_min": float(temp_min[i])
        })
    print(forecast_data)
    df = pd.DataFrame(forecast_data)
    print(df)
    return forecast_data

In [15]:
get_weather_forecast(lat, long, days=2)

[{'date': Timestamp('2026-03-16 23:00:00'), 'temp_max': 17.51850128173828, 'temp_min': 8.268499374389648}, {'date': Timestamp('2026-03-17 23:00:00'), 'temp_max': 14.768499374389648, 'temp_min': 8.668499946594238}]
                 date   temp_max  temp_min
0 2026-03-16 23:00:00  17.518501  8.268499
1 2026-03-17 23:00:00  14.768499  8.668500


[{'date': Timestamp('2026-03-16 23:00:00'),
  'temp_max': 17.51850128173828,
  'temp_min': 8.268499374389648},
 {'date': Timestamp('2026-03-17 23:00:00'),
  'temp_max': 14.768499374389648,
  'temp_min': 8.668499946594238}]

In [16]:
import pydantic
from pydantic import Field, BaseModel
from openai import pydantic_function_tool
import json

In [27]:
tools = []

In [28]:
class CityLatLongRequest(BaseModel):
    """Gets the latitude and longitude for a given city."""
    
    city: str = Field(..., description="Name of the city to get the latitude and longitude for")

class currentWeather(BaseModel):
    """Gets the current weather for a given city."""

    city: str = Field(..., description="Name of the city to get the current weather for")

class weatherForecast(BaseModel):
    """Gets the weather forecast for a given city."""

    city: str = Field(..., description="Name of the city to get the weather forecast for")

class exhangeRate(BaseModel):
    """Gets the exchange rate between two currencies."""

    from_currency: str = Field(..., description="The currency to convert from (e.g., 'USD')")
    to_currency: str = Field(..., description="The currency to convert to (e.g., 'EUR')")

class convertCurrency(BaseModel):
    """Converts an amount from one currency to another."""

    amount: float = Field(..., description="The amount of money to convert")
    from_currency: str = Field(..., description="The currency to convert from (e.g., 'USD')")
    to_currency: str = Field(..., description="The currency to convert to (e.g., 'EUR')")

class QueryUser(BaseModel):
    """
    A tool to ask questions to the user to get more information for the assistant to answer the question. Use this tool often to get more information from the user to answer their question better.
    The more specific the question, the better the answer will be. For example, if the user asks "What is the weather like?", you can ask "Which city are you interested in?" to get more information to answer the question better.
    """

    query: str = Field(..., description="Question to ask the user")


tools.append(pydantic_function_tool(CityLatLongRequest))
tools.append(pydantic_function_tool(currentWeather))
tools.append(pydantic_function_tool(weatherForecast))
tools.append(pydantic_function_tool(exhangeRate))
tools.append(pydantic_function_tool(convertCurrency))
tools.append(pydantic_function_tool(QueryUser))

In [29]:
tools

[{'type': 'function',
  'function': {'name': 'CityLatLongRequest',
   'strict': True,
   'parameters': {'description': 'Gets the latitude and longitude for a given city.',
    'properties': {'city': {'description': 'Name of the city to get the latitude and longitude for',
      'title': 'City',
      'type': 'string'}},
    'required': ['city'],
    'title': 'CityLatLongRequest',
    'type': 'object',
    'additionalProperties': False},
   'description': 'Gets the latitude and longitude for a given city.'}},
 {'type': 'function',
  'function': {'name': 'currentWeather',
   'strict': True,
   'parameters': {'description': 'Gets the current weather for a given city.',
    'properties': {'city': {'description': 'Name of the city to get the current weather for',
      'title': 'City',
      'type': 'string'}},
    'required': ['city'],
    'title': 'currentWeather',
    'type': 'object',
    'additionalProperties': False},
   'description': 'Gets the current weather for a given city.'}},
 

In [ ]:
def run_tool(tool_name, args):
    print(f"Running tool: {tool_name}")
    print(f"Arguments: {args}")
    match tool_name:
        case "CityLatLongRequest":
            return geocode_city(args["city"])
        case "currentWeather":
            lat, long = geocode_city(args["city"])
            return get_current_temperature(lat, long)
        case "weatherForecast":
            lat, long = geocode_city(args["city"])
            return get_weather_forecast(lat, long)
        case "exchangeRate":
            return get_exchange_rate(args["from_currency"], args["to_currency"])
        case "convertCurrency":
            return convert_currency(args["amount"], args["from_currency"], args["to_currency"])
        case "QueryUser":
            return input(args['query'])

        case _:
            return "No tool found"

In [33]:
from dotenv import load_dotenv
import os
from openai import AzureOpenAI

load_dotenv()

True

In [37]:
def llm_con_tools(input_text):
    turns = 10
    client = AzureOpenAI()

    systemprompt = """
    You are a helpful assistant that can provide information about the weather and geocode cities. You have access to the following tools:
    1. CityLatLongRequest: Gets the latitude and longitude for a given city. It takes one argument, "city", which is the name of the city to geocode.
    2. currentWeather: Gets the current weather for a given city. It takes one argument, "city", which is the name of the city to get the weather for.
    3. weatherForecast: Gets the weather forecast for a given city. It takes one argument, "city", which is the name of the city to get the weather forecast for.
    4. exhangeRate: Gets the exchange rate between two currencies. It takes two arguments, "from_currency" and "to_currency", which are the currencies to get the exchange rate for (e.g., "USD" and "EUR").
    5. convertCurrency: Converts an amount from one currency to another. It takes three arguments, "amount", "from_currency", and "to_currency", which are the amount of money to convert, the currency to convert from (e.g., "USD"), and the currency to convert to (e.g., "EUR").
    6. QueryUser: A tool to ask questions to the user to get more information for the assistant to answer the question. Use this tool often to get more information from the user to answer their question better. The more specific the question, the better the answer will be. For example, if the user asks "What is the weather like?", you can ask "Which city are you interested in?" to get more information to answer the question better.
    """ 

    messages=[
            {"role": "system", "content": systemprompt},
            {"role": "user", "content": input_text},
        ]

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        tools=tools,
    )
    # Detectamos si se ha usado una herramienta
    if response.choices[0].finish_reason == "tool_calls":
        for tool_call in response.choices[0].message.tool_calls:
            messages.append({"role": "assistant", "tool_calls": [tool_call]})
            args = json.loads(tool_call.function.arguments)
            tool_name = tool_call.function.name
            tool_result = run_tool(tool_name, args)
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": str(tool_result)})
        for turn in range(turns):
             response = client.chat.completions.create(
                model="gpt-4.1-mini",
                messages=messages,
                tools=tools,
            )
             if response.choices[0].finish_reason == "tool_calls":
                for tool_call in response.choices[0].message.tool_calls:
                    messages.append({"role": "assistant", "tool_calls": [tool_call]})
                    args = json.loads(tool_call.function.arguments)
                    tool_name = tool_call.function.name
                    tool_result = run_tool(tool_name, args)
                    messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": str(tool_result)})
             else:
                print("No tool used, final response:")
                return response.choices[0].message.content
        print(messages)
    return response.choices[0].message.content



pregunta = input("¿En que te puedo ayudar?")
print(llm_con_tools(pregunta))

Ejecutando herramienta: exhangeRate
Argumentos: {'from_currency': 'USD', 'to_currency': 'EUR'}
Exchange rate from USD to EUR: 0.8706
Ejecutando herramienta: convertCurrency
Argumentos: {'amount': 100, 'from_currency': 'USD', 'to_currency': 'EUR'}
100 USD is equal to 87.06 EUR at an exchange rate of 0.8706.
No tool used, final response:
100 US dollars is approximately 87.06 euros.
